In [ ]:
import sys
import os
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression, SGDClassifier

from sklearn.ensemble import StackingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix # type: ignore
from sklearn.preprocessing import Normalizer
sys.path.append(os.path.abspath("../../"))
from utility.preprocess import tokenizer, clean_eng
from utility.lexicon import LexiconTransformer
from sklearn.model_selection import StratifiedKFold
import matplotlib.pyplot as plt
import seaborn as sns

import json
import pandas as pd
import pickle



In [ ]:
with open("../../dataset/train_sentiment.json", "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)

In [ ]:
text = df['text']
sentiment = df['sentiment']

# text = df["text"].apply(clean_eng)

In [ ]:
text_train, text_test, sentiment_train, sentiment_test = train_test_split(
    text, sentiment, test_size=0.2, random_state=42
)

In [ ]:
base_models = [
    ("svm", LinearSVC(class_weight="balanced", random_state=42)),
    ("nb", MultinomialNB()),
    ("lr", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
    ("sgd", SGDClassifier(loss="log_loss", class_weight="balanced", random_state=42)),
    
]

meta_model = LogisticRegression(max_iter=1000, random_state=42)

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

stack_model = StackingClassifier(
    estimators=base_models,
    final_estimator=meta_model,
    cv=5
)

In [ ]:
word_vectorizer = TfidfVectorizer(
    tokenizer=tokenizer,
    ngram_range=(1, 2),
    sublinear_tf=True,
    max_df=0.9,
    min_df=1
)

char_vectorizer = TfidfVectorizer(
    analyzer='char_wb', 
    ngram_range=(1, 3), 
    sublinear_tf=True,
    max_df=0.9,
    min_df=3
)

combined_features = FeatureUnion([
    ("word_tfidf", word_vectorizer),
    ("char_tfidf", char_vectorizer),
    ("lexicon", LexiconTransformer())
], transformer_weights={
    "word_tfidf": 1.0,
    "char_tfidf": 0.5,
    "lexicon": 1.0
})

In [ ]:
pipeline = Pipeline([
    ('features', combined_features),
    ('norm', Normalizer()),
    ('classifier', stack_model)
])

pipeline.fit(text_train, sentiment_train)

In [ ]:
y_pred = pipeline.predict(text_test)

print("Accuracy:", accuracy_score(sentiment_test, y_pred))
print(classification_report(sentiment_test, y_pred))
# print(confusion_matrix(sentiment_test, y_pred))

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(
    sentiment_test,
    y_pred,
    cmap="Blues"
)

plt.title("Confusion Matrix")
plt.show()

In [ ]:
report = classification_report(sentiment_test, y_pred, output_dict=True)
df = pd.DataFrame(report).transpose()

df = df.iloc[:3]  # remove accuracy rows

df[["precision","recall","f1-score"]].plot(kind="bar", figsize=(8,5))
plt.title("Model Performance by Class")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.ylim(0,1)
plt.show()

In [ ]:
true_counts = pd.Series(sentiment_test).value_counts().sort_index()
pred_counts = pd.Series(y_pred).value_counts().sort_index()

df = pd.DataFrame({
    "True": true_counts,
    "Predicted": pred_counts
})

df.plot(kind="bar", figsize=(8,5))
plt.title("True vs Predicted Distribution")
plt.ylabel("Count")
plt.show()

In [ ]:
for text, true, pred in zip(text_test, sentiment_test, y_pred):
    if true != pred:
        print("TEXT:", text)
        print("TRUE:", true, "PRED:", pred)
        print()

In [ ]:

with open("../../models/weights/stack_model.pkl", "wb") as f:
    pickle.dump(pipeline, f)